# 0b · Build a Supply Chain Graph and Map It to the Warehouse Metadata

Run this **after** notebooks `0a`, `01` and `02`. The graph already knows *where* the warehouse data is (physical layer) and *what it means* (semantic layer). This notebook adds the third leg: a **native graph domain model**, a small manufacturing supply chain, and a **schema registry** that tells an agent how the two worlds line up.

```
FinishedGood -[:REQUIRES]-> Assembly -[:REQUIRES]-> Part -[:SOURCED_FROM]-> Supplier        (instances: real Snowflake keys)

GraphDatabase -> GraphNodeSchema / GraphRelSchema                                          (the model, described as data)
GraphNodeSchema -[:CORRELATES_TO | :ENRICHED_BY]-> Table / Column                          (class-level bridge to the warehouse)
BusinessTerm -[:STANDARDIZES]-> GraphNodeSchema                                            (shared vocabulary)
```

Steps:

1. Load configuration and check the layers from `01` / `02` are present
2. Pull real **seed keys** (parts, suppliers, costs) from Snowflake
3. Build the **micro bill of materials** in Neo4j
4. Register the graph's **own schema** as nodes
5. Bridge the schema to the Neocarta **tables, columns and business terms**
6. Verify, then run the **golden-thread lookup**: graph traversal to Snowflake SQL

> ⚠️ **Sections 3-5 write to Neo4j** (`NEO4J_DATABASE`). Snowflake is only read. Re-running is safe: every write is a `MERGE`.
>
> **Design rule:** part and supplier *instances* are never linked to warehouse tables. Only the *schema* nodes are, so the bridge says "a `Part` is looked up in `PART.P_PARTKEY`", not "this part is that row".

## 1 · Configuration

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
from neo4j import GraphDatabase

env_file = next((p for p in (Path(".env"),) if p.exists()), None)
assert env_file, "No .env found: copy .env.example to .env and fill it in (see docs/attendee_guide.md)"
load_dotenv(env_file, override=True)

PLACEHOLDERS = ("xxxxxxxx", "your-", "xy12345", "workshop_user")
required = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER",
            "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]
bad = [k for k in required if not os.getenv(k) or any(p in os.getenv(k) for p in PLACEHOLDERS)]
assert not bad, f"Missing or placeholder values in {env_file}: {bad}. Run notebook 0a first."

NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
SF_DATABASE, SF_SCHEMA = os.environ["SNOWFLAKE_DATABASE"], os.environ["SNOWFLAKE_SCHEMA"]
assert (SF_DATABASE.upper(), SF_SCHEMA.upper()) == ("SNOWFLAKE_SAMPLE_DATA", "TPCH_SF1"), \
    "This notebook seeds from the TPC-H sample tables PART / PARTSUPP / SUPPLIER."

# One timestamp for the whole run (not generated per row)
NOW = datetime.now(timezone.utc)
SESSION_TS = {"epoch": int(NOW.timestamp() * 1000), "iso": NOW.isoformat()}

driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))

def q(cypher, **params):
    # Run one Cypher statement in its own session; return rows as dicts
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(cypher, **params).data()

def snowflake_connect():
    # Same auth precedence as notebook 0a and neocarta: key-pair > authenticator > password
    kw = dict(account=os.getenv("SNOWFLAKE_ACCOUNT"), user=os.getenv("SNOWFLAKE_USER"),
              warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"), database=SF_DATABASE, schema=SF_SCHEMA,
              login_timeout=30, network_timeout=60)
    if os.getenv("SNOWFLAKE_ROLE"):
        kw["role"] = os.environ["SNOWFLAKE_ROLE"]
    if os.getenv("SNOWFLAKE_PRIVATE_KEY_PATH"):
        from cryptography.hazmat.primitives import serialization
        pw = os.getenv("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
        key = serialization.load_pem_private_key(Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).read_bytes(),
                                                 password=pw.encode() if pw else None)
        kw["private_key"] = key.private_bytes(serialization.Encoding.DER, serialization.PrivateFormat.PKCS8,
                                              serialization.NoEncryption())
    elif os.getenv("SNOWFLAKE_AUTHENTICATOR"):
        kw["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]
        if os.getenv("SNOWFLAKE_TOKEN"):
            kw["token"] = os.environ["SNOWFLAKE_TOKEN"]
    else:
        kw["password"] = os.getenv("SNOWFLAKE_PASSWORD")
    return snowflake.connector.connect(**kw)

sf = snowflake_connect()
print("Config file :", env_file)
print("Neo4j       :", os.environ["NEO4J_URI"], f"(database '{NEO4J_DATABASE}')")
print("Snowflake   :", f"{SF_DATABASE}.{SF_SCHEMA}")

### Check the layers from notebooks 01 and 02

The bridge in section 5 points at existing `Table` / `Column` / `BusinessTerm` nodes, so they must be there.

In [ ]:
NEEDED_TABLES = {"PART", "PARTSUPP", "SUPPLIER"}
have = {r["name"] for r in q("MATCH (:Schema {name: $s})-[:HAS_TABLE]->(t:Table) RETURN t.name AS name", s=SF_SCHEMA)}
missing = NEEDED_TABLES - have
if missing:
    raise SystemExit(f"❌ Tables {sorted(missing)} not in the graph. Run 01_load_metadata_into_graph.ipynb first.")
print(f"✅ Physical layer: {sorted(NEEDED_TABLES)} found")

terms = q("MATCH (b:BusinessTerm) RETURN count(b) AS n")[0]["n"]
if not terms:
    raise SystemExit("❌ No BusinessTerm nodes. Run 02_load_semantic_layer.ipynb first.")
print(f"✅ Semantic layer: {terms} business terms")

## 2 · Seed keys from Snowflake

The supply chain must use identifiers that really exist in the warehouse, so that a question answered in the graph can be finished with SQL later, with no key translation.

TPC-H has no bill of materials, so the *structure* (finished goods and assemblies) is invented below. The **parts, suppliers, names and supply costs are real rows** from `PART`, `SUPPLIER` and `PARTSUPP`.

Selection rule (deterministic): take three suppliers of part 1, then the 10 parts those suppliers supply, preferring parts that several of them supply, so some parts have a real choice of source.

In [ ]:
cur = sf.cursor()

cur.execute("""
    SELECT s.S_SUPPKEY, s.S_NAME, n.N_NAME, s.S_ACCTBAL
    FROM SUPPLIER s JOIN NATION n ON n.N_NATIONKEY = s.S_NATIONKEY
    WHERE s.S_SUPPKEY IN (SELECT PS_SUPPKEY FROM PARTSUPP WHERE PS_PARTKEY = 1 ORDER BY PS_SUPPKEY LIMIT 3)
    ORDER BY s.S_SUPPKEY
""")
suppliers = pd.DataFrame(cur.fetchall(), columns=["S_SUPPKEY", "S_NAME", "N_NAME", "S_ACCTBAL"])
assert len(suppliers) == 3, "expected three seed suppliers"
SUPP_KEYS = [int(k) for k in suppliers["S_SUPPKEY"]]

cur.execute(f"""
    SELECT PS_PARTKEY FROM PARTSUPP WHERE PS_SUPPKEY IN ({",".join(["%s"] * 3)})
    GROUP BY PS_PARTKEY ORDER BY COUNT(*) DESC, PS_PARTKEY LIMIT 10
""", SUPP_KEYS)
part_keys = [int(r[0]) for r in cur.fetchall()]
assert len(part_keys) == 10, "expected ten seed parts"

cur.execute(f"""
    SELECT p.P_PARTKEY, p.P_NAME, p.P_TYPE, ps.PS_SUPPKEY, ps.PS_SUPPLYCOST, ps.PS_AVAILQTY
    FROM PART p JOIN PARTSUPP ps ON ps.PS_PARTKEY = p.P_PARTKEY
    WHERE p.P_PARTKEY IN ({",".join(["%s"] * 10)}) AND ps.PS_SUPPKEY IN ({",".join(["%s"] * 3)})
    ORDER BY p.P_PARTKEY, ps.PS_SUPPLYCOST
""", part_keys + SUPP_KEYS)
supply = pd.DataFrame(cur.fetchall(), columns=["P_PARTKEY", "P_NAME", "P_TYPE", "PS_SUPPKEY", "PS_SUPPLYCOST", "PS_AVAILQTY"])
supply["PS_SUPPLYCOST"] = supply["PS_SUPPLYCOST"].astype(float)
parts = supply.drop_duplicates("P_PARTKEY")[["P_PARTKEY", "P_NAME", "P_TYPE"]].reset_index(drop=True)

print("Suppliers:")
display(suppliers)
print("Parts and who can supply them (cheapest first):")
display(supply)

## 3 · The micro bill of materials ✍️

Three finished goods, four assemblies (two are shared between products) and the ten parts, each assembly holding two or three parts. The shared assembly and the multi-sourced parts are deliberate: they make the traversal in section 6 more interesting than a tree.

The graph uses the warehouse keys as strings (`partKey: "1"`) and a readable SKU on top. Property types follow the model: `quantity` and `leadTimeWeeks` are integers, `unitCost` a float.

In [ ]:
FINISHED_GOODS = [
    {"sku": "FG-1000", "name": "Industrial Controller", "line": "Automation"},
    {"sku": "FG-2000", "name": "Telemetry Gateway", "line": "Connectivity"},
    {"sku": "FG-3000", "name": "Power Distribution Unit", "line": "Power"},
]
ASSEMBLIES = [
    {"sku": "ASM-1100", "name": "Control Board"},
    {"sku": "ASM-1200", "name": "Power Module"},
    {"sku": "ASM-2100", "name": "Comms Module"},
    {"sku": "ASM-3100", "name": "Breaker Panel"},
]
# (parent, child, quantity)
FG_REQUIRES = [("FG-1000", "ASM-1100", 1), ("FG-1000", "ASM-1200", 1),
               ("FG-2000", "ASM-1200", 2), ("FG-2000", "ASM-2100", 1),
               ("FG-3000", "ASM-2100", 1), ("FG-3000", "ASM-3100", 2)]
# Assembly -> indexes into the seeded part list (part 5 and part 0 are shared across assemblies)
ASM_PARTS = {"ASM-1100": [0, 1, 2], "ASM-1200": [3, 4, 5], "ASM-2100": [5, 6, 7], "ASM-3100": [8, 9, 0]}
ASM_REQUIRES = [(asm, int(parts.loc[i, "P_PARTKEY"]), 1 + (n % 3)) for asm, idx in ASM_PARTS.items() for n, i in enumerate(idx)]

PARTS = [{"partKey": str(int(r.P_PARTKEY)), "sku": f"P-{int(r.P_PARTKEY):06d}", "name": r.P_NAME,
          "leadTimeWeeks": 2 + int(r.P_PARTKEY) % 7}            # lead time is invented: TPC-H has none
         for r in parts.itertuples()]
SUPPLIERS = [{"supplierKey": str(int(r.S_SUPPKEY)), "name": r.S_NAME, "nation": r.N_NAME,
              "status": "ACTIVE" if float(r.S_ACCTBAL) >= 0 else "ON_HOLD"}   # derived from the real account balance
             for r in suppliers.itertuples()]
SOURCED_FROM = []
for pk, grp in supply.groupby("P_PARTKEY"):
    grp = grp.sort_values("PS_SUPPLYCOST")
    for n, r in enumerate(grp.itertuples()):
        SOURCED_FROM.append({"partKey": str(int(pk)), "supplierKey": str(int(r.PS_SUPPKEY)),
                             "isPrimary": n == 0, "unitCost": r.PS_SUPPLYCOST})   # cheapest source is primary
print(f"{len(FINISHED_GOODS)} finished goods, {len(ASSEMBLIES)} assemblies, {len(PARTS)} parts, "
      f"{len(SUPPLIERS)} suppliers, {len(SOURCED_FROM)} SOURCED_FROM edges")

In [ ]:
# Constraints first (each also creates the lookup index MERGE needs)
for label, prop in [("FinishedGood", "sku"), ("Assembly", "sku"), ("Part", "partKey"), ("Supplier", "supplierKey")]:
    q(f"CREATE CONSTRAINT {label.lower()}_{prop.lower()}_unique IF NOT EXISTS FOR (n:{label}) REQUIRE n.{prop} IS UNIQUE")

STAMP = {"now": SESSION_TS["epoch"], "iso": SESSION_TS["iso"]}

# Nodes: stable key in the MERGE, everything else via SET; created* only on create
q("""
UNWIND $rows AS r
MERGE (n:FinishedGood {sku: r.sku})
ON CREATE SET n.createdEpochMillis = $now, n.createdDatetime = datetime($iso)
ON MATCH SET n.modifiedEpochMillis = $now, n.modifiedDatetime = datetime($iso)
SET n.name = r.name, n.line = r.line
""", rows=FINISHED_GOODS, **STAMP)
q("""
UNWIND $rows AS r
MERGE (n:Assembly {sku: r.sku})
ON CREATE SET n.createdEpochMillis = $now, n.createdDatetime = datetime($iso)
ON MATCH SET n.modifiedEpochMillis = $now, n.modifiedDatetime = datetime($iso)
SET n.name = r.name
""", rows=ASSEMBLIES, **STAMP)
q("""
UNWIND $rows AS r
MERGE (n:Part {partKey: r.partKey})
ON CREATE SET n.createdEpochMillis = $now, n.createdDatetime = datetime($iso)
ON MATCH SET n.modifiedEpochMillis = $now, n.modifiedDatetime = datetime($iso)
SET n.sku = r.sku, n.name = r.name, n.leadTimeWeeks = r.leadTimeWeeks
""", rows=PARTS, **STAMP)
q("""
UNWIND $rows AS r
MERGE (n:Supplier {supplierKey: r.supplierKey})
ON CREATE SET n.createdEpochMillis = $now, n.createdDatetime = datetime($iso)
ON MATCH SET n.modifiedEpochMillis = $now, n.modifiedDatetime = datetime($iso)
SET n.name = r.name, n.nation = r.nation, n.status = r.status
""", rows=SUPPLIERS, **STAMP)

# Relationships
q("""
UNWIND $rows AS r
MATCH (a:FinishedGood {sku: r.parent}) MATCH (b:Assembly {sku: r.child})
MERGE (a)-[e:REQUIRES]->(b) SET e.quantity = r.quantity
""", rows=[{"parent": p, "child": c, "quantity": n} for p, c, n in FG_REQUIRES])
q("""
UNWIND $rows AS r
MATCH (a:Assembly {sku: r.parent}) MATCH (b:Part {partKey: r.child})
MERGE (a)-[e:REQUIRES]->(b) SET e.quantity = r.quantity
""", rows=[{"parent": a, "child": str(p), "quantity": n} for a, p, n in ASM_REQUIRES])
q("""
UNWIND $rows AS r
MATCH (p:Part {partKey: r.partKey}) MATCH (s:Supplier {supplierKey: r.supplierKey})
MERGE (p)-[e:SOURCED_FROM]->(s) SET e.isPrimary = r.isPrimary, e.unitCost = r.unitCost
""", rows=SOURCED_FROM)

pd.DataFrame(q("""
MATCH (n) WHERE n:FinishedGood OR n:Assembly OR n:Part OR n:Supplier
RETURN labels(n)[0] AS label, count(*) AS nodes ORDER BY label
"""))

Open it in Neo4j Workspace / Browser to see the shape:

```cypher
MATCH p = (:FinishedGood)-[:REQUIRES*1..2]->(:Part)-[:SOURCED_FROM]->(:Supplier) RETURN p LIMIT 50
```

## 4 · Describe the graph's own schema as data

An agent can ask a Neo4j database for its schema, but that only lists labels. Here the model is stored as **nodes you can annotate, link and search**: a `GraphDatabase` that contains one `GraphNodeSchema` per label (with its key property) and one `GraphRelSchema` per relationship type (with its end labels).

| Label | Meaning |
|---|---|
| `GraphDatabase` | The graph model as a whole (`SupplyChainTwin`) |
| `GraphNodeSchema` | One node label and the property that uniquely identifies it |
| `GraphRelSchema` | One relationship type, with `fromLabels` / `toLabels` |

In [ ]:
GRAPH_NAME = "SupplyChainTwin"
NODE_SCHEMAS = [
    {"label": "FinishedGood", "keyProperty": "sku", "description": "A product sold to customers"},
    {"label": "Assembly", "keyProperty": "sku", "description": "A sub-assembly built from parts"},
    {"label": "Part", "keyProperty": "partKey", "description": "A purchasable component, keyed on the warehouse part key"},
    {"label": "Supplier", "keyProperty": "supplierKey", "description": "A vendor, keyed on the warehouse supplier key"},
]
REL_SCHEMAS = [
    {"type": "REQUIRES", "fromLabels": ["FinishedGood", "Assembly"], "toLabels": ["Assembly", "Part"],
     "description": "Bill-of-materials edge; property: quantity"},
    {"type": "SOURCED_FROM", "fromLabels": ["Part"], "toLabels": ["Supplier"],
     "description": "A supplier can provide the part; properties: isPrimary, unitCost"},
]

for label, key in [("GraphDatabase", "name"), ("GraphNodeSchema", "id"), ("GraphRelSchema", "id")]:
    q(f"CREATE CONSTRAINT {label.lower()}_{key}_unique IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE")

# ids are scoped by graph name so a second model could coexist
q("""
MERGE (g:GraphDatabase {name: $graph})
ON CREATE SET g.createdEpochMillis = $now, g.createdDatetime = datetime($iso)
ON MATCH SET g.modifiedEpochMillis = $now, g.modifiedDatetime = datetime($iso)
SET g.description = 'Manufacturing supply chain slice: finished goods, assemblies, parts, suppliers'
WITH g
UNWIND $nodes AS r
MERGE (ns:GraphNodeSchema {id: $graph + '.' + r.label})
ON CREATE SET ns.createdEpochMillis = $now, ns.createdDatetime = datetime($iso)
ON MATCH SET ns.modifiedEpochMillis = $now, ns.modifiedDatetime = datetime($iso)
SET ns.label = r.label, ns.keyProperty = r.keyProperty, ns.description = r.description
MERGE (g)-[:CONTAINS_NODE_SCHEMA]->(ns)
""", graph=GRAPH_NAME, nodes=NODE_SCHEMAS, **STAMP)

q("""
MATCH (g:GraphDatabase {name: $graph})
UNWIND $rels AS r
MERGE (rs:GraphRelSchema {id: $graph + '.' + r.type})
ON CREATE SET rs.createdEpochMillis = $now, rs.createdDatetime = datetime($iso)
ON MATCH SET rs.modifiedEpochMillis = $now, rs.modifiedDatetime = datetime($iso)
SET rs.type = r.type, rs.fromLabels = r.fromLabels, rs.toLabels = r.toLabels, rs.description = r.description
MERGE (g)-[:CONTAINS_REL_SCHEMA]->(rs)
""", graph=GRAPH_NAME, rels=REL_SCHEMAS, **STAMP)

pd.DataFrame(q("""
MATCH (:GraphDatabase {name: $graph})-[:CONTAINS_NODE_SCHEMA]->(ns)
RETURN 'node' AS kind, ns.label AS name, ns.keyProperty AS key
UNION ALL
MATCH (:GraphDatabase {name: $graph})-[:CONTAINS_REL_SCHEMA]->(rs)
RETURN 'relationship' AS kind, rs.type AS name, rs.fromLabels[0] + ' ->' + rs.toLabels[0] AS key
""", graph=GRAPH_NAME))

## 5 · Bridge the schema to the warehouse and the vocabulary ✍️

Two new relationship types connect a `GraphNodeSchema` to the `Table` and `Column` nodes loaded in notebook 01:

| Relationship | Reads as | Properties |
|---|---|---|
| `CORRELATES_TO` | "this label's identity lives in this table/column" | `graphKey`, `relationalKey`, `joinType` |
| `ENRICHED_BY` | "to get more facts about this label, look up this table/column" | `metric`, `graphKey`, `relationalKey`, `lookupTable` |

and `BusinessTerm -[:STANDARDIZES]-> GraphNodeSchema` ties the shared vocabulary from notebook 02 to the graph model.

All of this is **class level**. No `Part` instance is connected to a warehouse row.

In [ ]:
def table_id(name): return f"{SF_DATABASE}.{SF_SCHEMA}.{name}".lower()
def column_id(table, col): return f"{table_id(table)}.{col.lower()}"

# (schema node id suffix, level, target table, target column, graphKey, relationalKey)
CORRELATES = [
    ("Part", "PART", "P_PARTKEY", "partKey", "P_PARTKEY"),
    ("Supplier", "SUPPLIER", "S_SUPPKEY", "supplierKey", "S_SUPPKEY"),
]
ENRICHES = [   # (schema, lookup table, relationalKey column, metric label, value column, graphKey)
    ("Part", "PARTSUPP", "PS_PARTKEY", "On-Hand Inventory", "PS_AVAILQTY", "partKey"),
    ("Part", "PARTSUPP", "PS_PARTKEY", "Supply Cost", "PS_SUPPLYCOST", "partKey"),
]

bridge_rows = []
for label, table, col, gkey, rkey in CORRELATES:
    bridge_rows.append({"schema": f"{GRAPH_NAME}.{label}", "tableId": table_id(table), "columnId": column_id(table, col),
                        "graphKey": gkey, "relationalKey": rkey})
q("""
UNWIND $rows AS r
MATCH (ns:GraphNodeSchema {id: r.schema})
MATCH (t:Table {id: r.tableId}) MATCH (c:Column {id: r.columnId})
MERGE (ns)-[a:CORRELATES_TO {target: 'table'}]->(t)
  SET a.graphKey = r.graphKey, a.relationalKey = r.relationalKey, a.joinType = 'EXACT_MATCH'
MERGE (ns)-[b:CORRELATES_TO {target: 'column'}]->(c)
  SET b.graphKey = r.graphKey, b.relationalKey = r.relationalKey, b.joinType = 'EXACT_MATCH'
""", rows=bridge_rows)

enrich_rows = [{"schema": f"{GRAPH_NAME}.{s}", "tableId": table_id(t), "columnId": column_id(t, vcol), "metric": metric,
                "graphKey": gk, "relationalKey": rk, "lookupTable": t, "valueColumn": vcol}
               for s, t, rk, metric, vcol, gk in ENRICHES]
q("""
UNWIND $rows AS r
MATCH (ns:GraphNodeSchema {id: r.schema})
MATCH (t:Table {id: r.tableId}) MATCH (c:Column {id: r.columnId})
MERGE (ns)-[a:ENRICHED_BY {metric: r.metric, target: 'table'}]->(t)
  SET a.graphKey = r.graphKey, a.relationalKey = r.relationalKey, a.lookupTable = r.lookupTable, a.valueColumn = r.valueColumn
MERGE (ns)-[b:ENRICHED_BY {metric: r.metric, target: 'column'}]->(c)
  SET b.graphKey = r.graphKey, b.relationalKey = r.relationalKey, b.lookupTable = r.lookupTable, b.valueColumn = r.valueColumn
""", rows=enrich_rows)

# The cost of a sourcing edge also lives in PARTSUPP, keyed on both ends
q("""
MATCH (rs:GraphRelSchema {id: $rs}) MATCH (t:Table {id: $tid}) MATCH (c:Column {id: $cid})
MERGE (rs)-[a:CORRELATES_TO {target: 'table'}]->(t)
  SET a.graphKey = 'partKey+supplierKey', a.relationalKey = 'PS_PARTKEY+PS_SUPPKEY', a.joinType = 'COMPOSITE_MATCH'
MERGE (rs)-[b:CORRELATES_TO {target: 'column'}]->(c)
  SET b.graphKey = 'unitCost', b.relationalKey = 'PS_SUPPLYCOST', b.joinType = 'PROPERTY_MATCH'
""", rs=f"{GRAPH_NAME}.SOURCED_FROM", tid=table_id("PARTSUPP"), cid=column_id("PARTSUPP", "PS_SUPPLYCOST"))

# Business terms from notebook 02 -> graph schema nodes
TERMS = {"Part": ["part id", "part name"], "Supplier": ["supplier id", "supplier name", "vendors"]}
term_rows = [{"schema": f"{GRAPH_NAME}.{s}", "term": t} for s, ts in TERMS.items() for t in ts]
linked = q("""
UNWIND $rows AS r
MATCH (ns:GraphNodeSchema {id: r.schema}) MATCH (b:BusinessTerm) WHERE toLower(b.name) = r.term
MERGE (b)-[:STANDARDIZES]->(ns)
RETURN ns.label AS label, b.name AS term
""", rows=term_rows)
missing_terms = {r["term"] for r in term_rows} - {r["term"].lower() for r in linked}
print(f"✅ {len(linked)} STANDARDIZES links", "" if not missing_terms else f"⚠️ no BusinessTerm named: {sorted(missing_terms)}")

## 6 · Verify the bridge

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append(ok)
    print(("✅" if ok else "❌"), name, ("— " + detail) if detail else "")

c = {r["label"]: r["n"] for r in q("MATCH (n) WHERE n:FinishedGood OR n:Assembly OR n:Part OR n:Supplier RETURN labels(n)[0] AS label, count(*) AS n")}
check("Instance nodes", c == {"FinishedGood": 3, "Assembly": 4, "Part": 10, "Supplier": 3}, str(c))
check("Every part is sourced", q("MATCH (p:Part) WHERE NOT (p)-[:SOURCED_FROM]->() RETURN count(p) AS n")[0]["n"] == 0)
check("Every part is in an assembly, every finished good reaches parts",
      q("MATCH (p:Part) WHERE NOT (:Assembly)-[:REQUIRES]->(p) RETURN count(p) AS n")[0]["n"] == 0
      and q("MATCH (f:FinishedGood) WHERE NOT (f)-[:REQUIRES*2]->(:Part) RETURN count(f) AS n")[0]["n"] == 0)
check("Graph schema registry", q("""MATCH (:GraphDatabase {name: $g})-[:CONTAINS_NODE_SCHEMA]->(n) WITH count(n) AS nodes
                                    MATCH (:GraphDatabase {name: $g})-[:CONTAINS_REL_SCHEMA]->(r) RETURN nodes, count(r) AS rels""", g=GRAPH_NAME)[0] == {"nodes": 4, "rels": 2})
check("Each primary source is the cheapest",
      q("""MATCH (p:Part)-[e:SOURCED_FROM]->() WITH p, collect(e) AS es
           WHERE size([x IN es WHERE x.isPrimary]) <> 1 RETURN count(p) AS n""")[0]["n"] == 0)

# Design rule: nothing at instance level touches the warehouse metadata
leak = q("""MATCH (n)-[r]-(m) WHERE (n:FinishedGood OR n:Assembly OR n:Part OR n:Supplier) AND (m:Table OR m:Column OR m:Metric)
            RETURN count(r) AS n""")[0]["n"]
check("No instance-to-table edges (class level only)", leak == 0, f"{leak} found")

# Key coherence: every graph key exists in the warehouse
cur.execute("SELECT COUNT(*) FROM PART WHERE P_PARTKEY IN (" + ",".join(["%s"] * len(PARTS)) + ")", [int(p["partKey"]) for p in PARTS])
check("All part keys exist in Snowflake PART", cur.fetchone()[0] == len(PARTS))
cur.execute("SELECT COUNT(*) FROM SUPPLIER WHERE S_SUPPKEY IN (" + ",".join(["%s"] * len(SUPPLIERS)) + ")", [int(s["supplierKey"]) for s in SUPPLIERS])
check("All supplier keys exist in Snowflake SUPPLIER", cur.fetchone()[0] == len(SUPPLIERS))

pd.DataFrame(q("""
MATCH (ns:GraphNodeSchema)-[r:CORRELATES_TO|ENRICHED_BY]->(x)
WHERE r.target = 'table'
RETURN ns.label AS graph_label, type(r) AS bridge, coalesce(r.metric, '') AS metric, x.name AS table, r.graphKey AS graph_key, r.relationalKey AS warehouse_key
UNION ALL
MATCH (rs:GraphRelSchema)-[r:CORRELATES_TO]->(x:Table)
RETURN rs.type AS graph_label, type(r) AS bridge, '' AS metric, x.name AS table, r.graphKey AS graph_key, r.relationalKey AS warehouse_key
"""))

## 7 · The golden thread: from a graph question to real SQL

> *"Supplier X has a problem. Which products are affected, and what stock do we hold of the parts they supply?"*

1. **Graph:** traverse `Supplier <- Part <- Assembly <- FinishedGood` to find the affected products.
2. **Schema bridge:** ask the `Part` schema node where stock lives. It answers with table, key column and value column, from `ENRICHED_BY`. Nothing about `PARTSUPP` is hard coded below.
3. **Snowflake:** build the SQL from those coordinates and run it against the real keys. `PARTSUPP` holds one row per part and supplier, so the figure is the quantity available across **all** of a part's suppliers.

In [ ]:
SUPPLIER_IN_TROUBLE = SUPPLIERS[0]["supplierKey"]

affected = q("""
MATCH (s:Supplier {supplierKey: $sk})<-[src:SOURCED_FROM]-(p:Part)<-[:REQUIRES*1..2]-(f:FinishedGood)
RETURN DISTINCT f.sku AS finished_good, f.name AS product, p.partKey AS part_key, src.isPrimary AS is_primary_source
ORDER BY finished_good, part_key
""", sk=SUPPLIER_IN_TROUBLE)
affected_df = pd.DataFrame(affected)
check("Disruption query finds affected finished goods", len(affected_df) > 0, f"{affected_df['finished_good'].nunique()} products, {affected_df['part_key'].nunique()} parts")
display(affected_df)

In [ ]:
# 2. Ask the schema bridge where the on-hand inventory lives
coords = q("""
MATCH (:GraphNodeSchema {label: 'Part'})-[e:ENRICHED_BY {metric: 'On-Hand Inventory', target: 'table'}]->(t:Table)
MATCH (:GraphNodeSchema {label: 'Part'})-[:ENRICHED_BY {metric: 'On-Hand Inventory', target: 'column'}]->(v:Column)
RETURN t.id AS table_id, t.name AS table, e.relationalKey AS key_column, v.name AS value_column
""")[0]
check("Bridge returns Snowflake coordinates for Part inventory", coords["table"] == "PARTSUPP" and coords["value_column"] == "PS_AVAILQTY", str(coords))

# 3. Build and run SQL from those coordinates (identifiers come from the graph, values are bound parameters)
part_keys_hit = sorted({int(k) for k in affected_df["part_key"]})
sql = (f'SELECT "{coords["key_column"]}" AS part_key, SUM("{coords["value_column"]}") AS total_available '
       f'FROM {coords["table_id"].upper()} WHERE "{coords["key_column"]}" IN ({",".join(["%s"] * len(part_keys_hit))}) GROUP BY 1 ORDER BY 1')
print(sql, "\n")
cur.execute(sql, part_keys_hit)
stock = pd.DataFrame(cur.fetchall(), columns=["part_key", "total_available"])
check("Real SQL returned stock for the affected parts", len(stock) == len(part_keys_hit), f"{len(stock)} of {len(part_keys_hit)} parts")
display(stock)

## Summary and next steps

The graph now holds three connected layers:

| Layer | Notebook | Answers |
|---|---|---|
| Physical | 01 | *Where* is the data? |
| Semantic | 02 | *What does it mean*, how do tables join? |
| Domain graph + schema bridge | 0b | *How do the things we care about connect*, and which table/column holds more facts about each? |

Next: let an agent traverse the supply chain in the graph, read the `ENRICHED_BY` coordinates and write the Snowflake query itself, the same thread you just walked by hand.

In [ ]:
sf.close()
driver.close()